# SLU Mocap
**Runtime › Change runtime type › T4 GPU**, then **Runtime › Run all**.

Reads `My Drive/SLU_Mocap/inbox`, writes `My Drive/SLU_Mocap/out` (BVH, FBX, contact sheets, `qc_report.html`). Already-processed takes are skipped.

In [ ]:
#@title 1 · Connect Drive + install (≈2 min first time, cached after)
WATCH = False  #@param {type:"boolean"}
LEGACY_FOLDER = ""  #@param {type:"string"}
from google.colab import drive; drive.mount('/content/drive')
import os, subprocess, pathlib
ROOT = '/content/drive/MyDrive/SLU_Mocap'
for d in ('inbox','out','cache','calib','models','review'): os.makedirs(f'{ROOT}/{d}', exist_ok=True)
if not os.path.exists('/content/slu-mocap'):
    subprocess.run(['git','clone','-q','--depth','1','https://github.com/mikeylambo/slu-mocap','/content/slu-mocap'], check=True)
else:
    subprocess.run(['git','-C','/content/slu-mocap','pull','-q'])
%pip -q install rtmlib onnxruntime-gpu opencv-contrib-python-headless pyyaml scipy
# cache RTMW / YOLOX weights on Drive
cache = pathlib.Path.home()/'.cache'/'rtmlib'
cache.parent.mkdir(parents=True, exist_ok=True)
if not cache.is_symlink():
    subprocess.run(['rm','-rf',str(cache)]); cache.symlink_to(f'{ROOT}/models')
# Blender (headless) for FBX export, cached on Drive
BL = f'{ROOT}/models/blender-4.2.3-linux-x64'
if not os.path.exists(BL):
    subprocess.run(f'wget -q https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz -O /tmp/b.tar.xz && tar -xf /tmp/b.tar.xz -C {ROOT}/models', shell=True)
os.makedirs('/content/blender', exist_ok=True)
if not os.path.exists('/content/blender/blender'): os.symlink(f'{BL}/blender', '/content/blender/blender')
%cd /content/slu-mocap
print('ready')

In [ ]:
#@title 2 · Import legacy clips (optional: plain camera clips with a clap + spoken move name)
if LEGACY_FOLDER:
    %pip -q install openai-whisper
    !python -m pipeline.ingest --root "$ROOT" --legacy "$LEGACY_FOLDER" 

In [ ]:
#@title 3 · Run all (set WATCH to keep processing new uploads)
!python -m pipeline.run --root "$ROOT" {'--watch' if WATCH else ''}

In [ ]:
#@title 4 · QC report
from IPython.display import HTML, display
display(HTML(open(f'{ROOT}/out/qc_report.html').read()))